### Script per generare il file 'followers.csv' a partire dai tre file:
  - ai_all_followers.csv
  - chatgpt_all_followers.csv
  - ml_all_followers.csv

### Obiettivi:
1. 50000 archi.
3. Favorire archi che coinvolgono nodi di alto grado su entrambi i lati.
4. Includere **obbligatoriamente** i thread_user_pk (da ciascun file data seed):


In [16]:
from pathlib import Path
DATA_DIR = Path.cwd().parent.parent / 'data' / 'interim' 
DATA_DIR
PATH_AI = DATA_DIR / 'ai_all_followers.csv'
PATH_CHATGPT = DATA_DIR / 'chatgpt_all_followers.csv'
PATH_ML = DATA_DIR / 'ml_all_followers.csv'
PATH_OUTPUT = DATA_DIR.parent / 'processed' /'followers.csv'

In [17]:
import pandas as pd
mandatory_ai = set(pd.read_csv(r"c:\\Users\\pasqua\\Desktop\\univ\\progettoasnm\\Code\data_extraction\data\raw\AI\data.csv")["thread_user_pk"].to_list())
mandatory_chatgpt = set(pd.read_csv(r"c:\\Users\\pasqua\\Desktop\\univ\\progettoasnm\\Code\data_extraction\data\raw\ChatGPT\data.csv")["user_threads_userpk"].to_list())
mandatory_ml = set(pd.read_csv(r"c:\\Users\\pasqua\\Desktop\\univ\\progettoasnm\\Code\data_extraction\data\raw\ML\data.csv")["user_pk"].to_list())

In [18]:
import pandas as pd

def main():
    INPUT_PATHS = [
        PATH_AI,
        PATH_CHATGPT,
        PATH_ML
    ]
    mandatory_nodes = mandatory_ai | mandatory_chatgpt | mandatory_ml
    
    # --- Parametri ---
    TARGET_EDGES = 50000
    RAND = 42
    
    # -- 1) Carico e concateno --
    dfs = [pd.read_csv(p) for p in INPUT_PATHS]
    df = pd.concat(dfs, ignore_index=True)
    df = df.drop(columns=[c for c in df.columns if "count" in c.lower()], errors="ignore")

    U = "thread_user_pk"
    F = "thread_follower_pk"

    # --- Rimuovo duplicati iniziali ---
    df = df.drop_duplicates(subset=[U, F], keep='first')

    # --- 2) Calcolo grado e peso ---
    deg_u = df[U].value_counts().rename("deg_u")
    deg_f = df[F].value_counts().rename("deg_f")
    df = df.merge(deg_u, left_on=U, right_index=True)
    df = df.merge(deg_f, left_on=F, right_index=True)
    df["weight"] = df["deg_u"] + df["deg_f"]

    # --- 3) Archi obbligatori ---
    selected_indices = set()
    for node in mandatory_nodes:
        idxs = df.index[df[U] == node].tolist()
        if not idxs:
            continue
        best = df.loc[idxs, "weight"].idxmax()
        selected_indices.add(best)
    print(f"Inclusi archi obbligatori: {len(selected_indices)}")

    # Inizializzo set dei nodi selezionati
    nodes = set(df.loc[list(selected_indices), U]) | set(df.loc[list(selected_indices), F])

    # --- 4) Selezione greedy per minimizzare nuovi nodi ---
    rest = df.drop(index=list(selected_indices)).copy()
    # Calcolo quanti nuovi nodi introduce ogni arco
    rest['new_nodes'] = rest.apply(lambda row: len({row[U], row[F]} - nodes), axis=1)
    # Ordino per meno nuovi nodi e poi per peso decrescente
    rest_sorted = rest.sort_values(['new_nodes', 'weight'], ascending=[True, False])
    need = TARGET_EDGES - len(selected_indices)
    take = rest_sorted.head(need)
    selected_indices.update(take.index.tolist())
    # Aggiungo tutti i nodi introdotti
    new_endpoints = set(take[U]).union(set(take[F]))
    nodes |= new_endpoints

    # --- 5) Salvo e check finale ---
    out_df = df.loc[list(selected_indices)].sample(frac=1, random_state=RAND)
    total_edges = len(out_df)
    total_nodes = out_df[U].nunique() + out_df[F].nunique()
    print(f"Edges: {total_edges}, Nodes: {total_nodes}, Ratio: {total_edges/total_nodes:.3f}")

    out_df = out_df.drop(columns=["deg_u", "deg_f", "weight"], errors="ignore")
    out_df.to_csv(PATH_OUTPUT, index=False)
    print(f"👉 '{PATH_OUTPUT}' creato con successo.")

if __name__ == "__main__":
    main()

Inclusi archi obbligatori: 81
Edges: 50000, Nodes: 45539, Ratio: 1.098
👉 'c:\Users\pasqua\Desktop\univ\progettoasnm\Code\data_extraction\data\processed\followers.csv' creato con successo.
